# Functions only

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# ============================================================================
# GLOBALS / ENVIRONMENT
# ============================================================================

ALLOWED_DATASETS = ("STU00225089",)
DATASET_NAME_CATATONIA = "STU00225089"
_APARC_MEASURES = {"volume", "thickness", "meancurv"}


def _require_env_path(name: str) -> Path:
    value = os.getenv(name)

    if not value:
        raise EnvironmentError(f"Required environment variable '{name}' is not set.")

    return Path(value)


def add_base_subjid(df: pd.DataFrame, source_col: str = "subjid") -> pd.DataFrame:
    
    df = df.copy()

    if source_col not in df.columns:
        raise KeyError(f"Required subject-ID column '{source_col}' is missing.")

    df[source_col] = df[source_col].astype("string").str.strip()

    if df[source_col].isna().any():
        bad_rows = df.index[df[source_col].isna()].tolist()
        raise ValueError(f"Missing subject IDs in column '{source_col}' at rows: {bad_rows}")

    if df[source_col].eq("").any():
        bad_rows = df.index[df[source_col].eq("")].tolist()
        raise ValueError(f"Empty subject IDs in column '{source_col}' at rows: {bad_rows}")

    df["subjid_base"] = df[source_col].str.replace(r"-\d{3}$", "", regex=True)

    return df


def load_aparcaseg_table(fn: str) -> pd.DataFrame:
    """Load an aparc+aseg E-field stats table while preserving modeled IDs."""

    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"aparc+aseg stats table does not exist: {path}")

    df = pd.read_csv(path, sep="\t")
    
    if "subject_id" in df.columns and "subjid" not in df.columns:
        df = df.rename(columns={"subject_id": "subjid"})

    if "subjid" not in df.columns:
        raise KeyError("Expected either 'subjid' or 'subject_id' column.")

    df = add_base_subjid(df, source_col="subjid")

    duplicate_mask = df["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Duplicate modeled subject IDs found in {path}: {duplicates}")

    roi_cols = [col for col in df.columns if col not in {"subjid", "subjid_base"}]
    df[roi_cols] = df[roi_cols].apply(pd.to_numeric, errors="raise")

    return df


def get_stats_table_files(
    results_dir: str,
    field_name: str = "magnE_mean",
    measure: str = "mean",
    static_step: int = 0,
    adaptive_step: int = 2) -> dict:

    results_dir = Path(results_dir)

    if not results_dir.is_dir():
        raise NotADirectoryError(f"Results directory does not exist: {results_dir}")

    filename = f"aparc+aseg.{field_name}.{measure}.stats.table"
    files = {}

    for model in ("skin_single", "skin_double"):
        model_dir = results_dir / model / "stats"
        static_file = model_dir / f"step_{static_step}" / filename
        adaptive_file = model_dir / f"step_{adaptive_step}" / filename

        missing_files = [path for path in (static_file, adaptive_file) if not path.is_file()]

        if missing_files:
            raise FileNotFoundError("Missing expected stats table(s):\n" + "\n".join(str(path) for path in missing_files))

        files[model] = {
            "static": str(static_file),
            "adaptive": str(adaptive_file),
        }

    return files


def load_stimulus(filepath: Path = None) -> pd.DataFrame:
    """
    Currently unused in the main dataframe build.
    Retained for later need-based merging.
    """

    if filepath is None:
        raise ValueError("filepath must be provided.")

    filepath = Path(filepath)

    if not filepath.is_file():
        raise FileNotFoundError(f"Stimulus file does not exist: {filepath}")

    df_stim = pd.read_excel(filepath, sheet_name="stimulus")

    required_cols = ["subjid", "date", "pulse_width_ms", "frequency_hz", "percent_charge"]
    missing_cols = [col for col in required_cols if col not in df_stim.columns]

    if missing_cols:
        raise KeyError(f"Stimulus table is missing required columns: {missing_cols}")

    col = "subjid"
    idx_bad_rows = df_stim.index[df_stim[col].isna()].tolist()

    print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{col}': {idx_bad_rows}")

    df_stim = df_stim.drop(index=idx_bad_rows).copy()
    df_stim["subjid_org"] = df_stim["subjid"].copy()
    df_stim = add_base_subjid(df_stim, source_col="subjid")

    df_stim["pulse_width_ms"] = df_stim["pulse_width_ms"].replace({
        5: 0.5,
        25: 0.25,
        50: 0.50,
        75: 0.75,
    })

    return df_stim


def load_cogscores_batch(fn: str) -> pd.DataFrame:
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)

    cleaned_columns = df.columns.str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = ["subjid", "cgi_start", "cgi_end", "date_start", "date_end (acute)", "mrn"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    for column in ("cgi_start", "cgi_end"):
        if df[column].isna().any():
            idx_bad_rows = df.index[df[column].isna()].tolist()
            print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{column}': {idx_bad_rows}")
            df = df.drop(index=idx_bad_rows).copy()

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")
    df = df.drop(columns=["mrn"])

    return df


def df_combine_stim_cog(
    df_stim: pd.DataFrame,
    df_cog: pd.DataFrame,
    output_path: str | None = "data_model.csv") -> pd.DataFrame:
    """
    Currently unused in the main dataframe build.
    Retained for later need-based merging.
    """

    df_stim = df_stim.copy()
    df_cog = df_cog.copy()

    required_stim_cols = {"subjid", "subjid_base", "date"}
    required_cog_cols = {"subjid", "subjid_base", "date_start", "date_end (acute)"}

    missing_stim = required_stim_cols - set(df_stim.columns)
    missing_cog = required_cog_cols - set(df_cog.columns)

    if missing_stim:
        raise KeyError(f"df_stim missing required columns: {sorted(missing_stim)}")

    if missing_cog:
        raise KeyError(f"df_cog missing required columns: {sorted(missing_cog)}")

    stim_subjects = set(df_stim["subjid_base"].dropna())
    cog_subjects = set(df_cog["subjid_base"].dropna())

    only_stim = sorted(stim_subjects - cog_subjects)
    only_cog = sorted(cog_subjects - stim_subjects)

    if only_stim or only_cog:
        raise ValueError(
            "Subject mismatch between stimulus and cognitive data.\n"
            f"Only in stimulus ({len(only_stim)}): {only_stim}\n"
            f"Only in cognitive ({len(only_cog)}): {only_cog}"
        )

    date_columns = [
        (df_stim, "date"),
        (df_cog, "date_start"),
        (df_cog, "date_end (acute)"),
    ]

    for dataframe, column in date_columns:
        if dataframe[column].isna().any():
            bad_rows = dataframe.index[dataframe[column].isna()].tolist()
            raise ValueError(f"Missing date values in '{column}' at rows: {bad_rows}")

        dataframe[column] = pd.to_datetime(dataframe[column], format="mixed", errors="raise").dt.normalize()

    df_model = df_stim.merge(
        df_cog,
        on="subjid_base",
        how="inner",
        suffixes=("_stim", "_cog"),
        validate="many_to_many",
    )

    if df_model.empty:
        raise ValueError("Stimulus/cognitive merge produced zero rows.")

    in_window = df_model["date"].between(df_model["date_start"], df_model["date_end (acute)"], inclusive="both")

    n_before = len(df_model)
    n_outside = int((~in_window).sum())

    df_model = df_model.loc[in_window].copy()
    df_model = df_model.sort_values(["subjid_base", "date"]).reset_index(drop=True)

    if df_model.empty:
        raise ValueError("No stimulation sessions remain after applying the acute-treatment date windows.")

    if n_outside > 0:
        print(
            f"df_combine_stim_cog: intentionally excluded {n_outside}/{n_before} merged row combinations "
            "because the stimulation date was outside the corresponding acute-treatment window."
        )

    df_model = df_model.replace(r"^\s*$", np.nan, regex=True)
    df_model["cgi"] = np.nan

    start_mask = df_model["date"].eq(df_model["date_start"])
    end_mask = df_model["date"].eq(df_model["date_end (acute)"])

    df_model.loc[start_mask, "cgi"] = df_model.loc[start_mask, "cgi_start"]
    df_model.loc[end_mask, "cgi"] = df_model.loc[end_mask, "cgi_end"]

    if output_path is not None:
        df_model.to_csv(output_path, index=False)

    return df_model

def df_compute_ef_sf(df: pd.DataFrame=None) -> pd.DataFrame:
    """ef_sf = pulse_width_ms * frequency_hz * percent_charge."""

    df = df.copy()

    sf_columns = ["pulse_width_ms", "frequency_hz", "percent_charge"]
    missing_cols = [col for col in sf_columns if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing EF scaling-factor columns: {missing_cols}")

    df[sf_columns] = df[sf_columns].apply(pd.to_numeric, errors="raise")

    if df[sf_columns].isna().any().any():
        bad_rows = df.index[df[sf_columns].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing values found in EF scaling-factor columns at rows: {bad_rows}")

    df["ef_sf"] = df["pulse_width_ms"] * df["frequency_hz"] * df["percent_charge"]

    return df


def df_compute_change_cgi(df: pd.DataFrame) -> pd.DataFrame:
    """Compute cgi_change = cgi_end - cgi_start."""

    required_cols = ["subjid", "cgi_start", "cgi_end", "response_type", "session_num"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Required columns are missing: {missing_cols}")

    result = df.loc[:, required_cols].copy()

    result["cgi_start"] = pd.to_numeric(result["cgi_start"], errors="raise")
    result["cgi_end"] = pd.to_numeric(result["cgi_end"], errors="raise")

    if result[["cgi_start", "cgi_end"]].isna().any().any():
        raise ValueError("Missing CGI start or end values detected.")

    result["cgi_change"] = result["cgi_end"] - result["cgi_start"]

    return result


def add_roi_stats(df_model: pd.DataFrame, df_stats: pd.DataFrame) -> pd.DataFrame:
    """
    Attach clinical/person-level information to every modeled MRI/FEM scan.

    df_stats is the primary dataframe so modeled subjects are never lost
    simply because clinical data are unavailable.

    subjid:
        exact modeled MRI/FEM ID

    subjid_base:
        person-level ID

    subjid_cog:
        original ID from cognitive-score table
    """

    df_model = df_model.copy()
    df_stats = df_stats.copy()

    required_model_cols = {"subjid_base"}
    required_stats_cols = {"subjid", "subjid_base"}

    missing_model = required_model_cols - set(df_model.columns)
    missing_stats = required_stats_cols - set(df_stats.columns)

    if missing_model:
        raise KeyError(f"df_model missing required columns: {sorted(missing_model)}")

    if missing_stats:
        raise KeyError(f"df_stats missing required columns: {sorted(missing_stats)}")

    duplicate_mask = df_stats["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df_stats.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Modeled subject IDs must be unique within an E-field stats table. Duplicates: {duplicates}")

    modeled_subjects = set(df_stats["subjid_base"].dropna())
    clinical_subjects = set(df_model["subjid_base"].dropna())

    modeled_without_clinical = sorted(modeled_subjects - clinical_subjects)
    clinical_without_model = sorted(clinical_subjects - modeled_subjects)

    if modeled_without_clinical:
        print(
            f"add_roi_stats: {len(modeled_without_clinical)} modeled clinical subjects have no cognitive-score data: "
            f"{modeled_without_clinical}"
        )

    if clinical_without_model:
        print(
            f"add_roi_stats: {len(clinical_without_model)} cognitive-score subjects have no model in this dataset: "
            f"{clinical_without_model}"
        )

    if "subjid" in df_model.columns:
        df_model = df_model.rename(columns={"subjid": "subjid_cog"})

    clinical_counts = df_model.groupby("subjid_base").size()
    modeled_counts = df_stats.groupby("subjid_base").size()

    expected_rows = 0

    for subject in modeled_subjects:
        n_models = int(modeled_counts[subject])
        n_clinical = int(clinical_counts[subject]) if subject in clinical_counts.index else 1
        expected_rows += n_models * n_clinical

    result = df_stats.merge(
        df_model,
        on="subjid_base",
        how="left",
        validate="many_to_many",
        suffixes=("", "_cog"),
    )

    if len(result) != expected_rows:
        raise RuntimeError(
            f"Unexpected row count after modeled-to-clinical merge. "
            f"Expected: {expected_rows}, observed: {len(result)}"
        )

    return result


def build_model_roi(df_model: pd.DataFrame, 
                    files_agg: dict, 
                    skin_model: str = "skin_single", 
                    model_type: str = "static") -> pd.DataFrame:

    valid_skin_models = {"skin_single", "skin_double"}
    valid_model_types = {"static", "adaptive"}

    if skin_model not in valid_skin_models:
        raise ValueError(f"skin_model must be one of {sorted(valid_skin_models)}.")

    if model_type not in valid_model_types:
        raise ValueError(f"model_type must be one of {sorted(valid_model_types)}.")

    dataset_types = {
        "org": False,
        "synthsr": True,
    }

    dfs = []

    for dataset, is_synth in dataset_types.items():
        if dataset not in files_agg:
            raise KeyError(f"Dataset '{dataset}' missing from files_agg.")

        if skin_model not in files_agg[dataset]:
            raise KeyError(f"Skin model '{skin_model}' missing from files_agg['{dataset}'].")

        if model_type not in files_agg[dataset][skin_model]:
            raise KeyError(f"Model type '{model_type}' missing from files_agg['{dataset}']['{skin_model}'].")

        fn = files_agg[dataset][skin_model][model_type]

        print(f"\nLoading {'SynthSR' if is_synth else 'original'} {skin_model}/{model_type}: ({fn})")

        df_stats = load_aparcaseg_table(fn)

        print(f"  Modeled scans: {df_stats['subjid'].nunique()}")
        print(f"  Clinical subjects represented: {df_stats['subjid_base'].nunique()}")

        df_model_dataset = add_roi_stats(df_model=df_model, df_stats=df_stats)
        df_model_dataset["is_synth"] = bool(is_synth)

        dfs.append(df_model_dataset)

    if not dfs:
        raise RuntimeError("No modeling datasets were loaded.")

    df_model_roi = pd.concat(dfs, ignore_index=True)
    df_model_roi["is_synth"] = df_model_roi["is_synth"].astype(bool)

    return df_model_roi


def add_subject_info(
    df: pd.DataFrame,
    df_subject_info: pd.DataFrame,
    age_aggregation: str = "min") -> pd.DataFrame:

    df = df.copy()
    df_subject_info = df_subject_info.copy()

    if "subjid_base" not in df.columns:
        raise KeyError("df must contain 'subjid_base'.")

    if "subjid_base" not in df_subject_info.columns:
        if "subjid" not in df_subject_info.columns:
            raise KeyError("df_subject_info must contain either 'subjid_base' or 'subjid'.")

        df_subject_info = add_base_subjid(df_subject_info, source_col="subjid")

    required_cols = ["subjid_base", "age", "sex"]
    missing_cols = [col for col in required_cols if col not in df_subject_info.columns]

    if missing_cols:
        raise KeyError(f"Subject info missing required columns: {missing_cols}")

    if age_aggregation not in {"min", "max", "mean", "first"}:
        raise ValueError("`age_aggregation` must be 'min', 'max', 'mean', or 'first'.")

    subject_info = df_subject_info[required_cols].copy()

    for column in ("subjid_base", "age", "sex"):
        if subject_info[column].isna().any():
            bad_rows = subject_info.index[subject_info[column].isna()].tolist()
            raise ValueError(f"Missing '{column}' values in subject info at rows: {bad_rows}")

    subject_info["age"] = pd.to_numeric(subject_info["age"], errors="raise")

    sex_conflicts = subject_info.groupby("subjid_base")["sex"].nunique(dropna=False).gt(1)
    conflicting_subjects = sex_conflicts[sex_conflicts].index.tolist()

    if conflicting_subjects:
        raise ValueError(f"Conflicting sex values found for clinical subjects: {conflicting_subjects}")

    age_by_subject = subject_info.groupby("subjid_base")["age"].agg(age_aggregation).rename("age")
    sex_by_subject = subject_info.groupby("subjid_base")["sex"].first().rename("sex")

    subject_info_unique = pd.concat([age_by_subject, sex_by_subject], axis=1).reset_index()

    modeled_subjects = set(df["subjid_base"].dropna())
    info_subjects = set(subject_info_unique["subjid_base"].dropna())

    missing_info = sorted(modeled_subjects - info_subjects)

    if missing_info:
        raise ValueError(f"Missing subject-info records for clinical subjects: {missing_info}")

    return df.merge(subject_info_unique, on="subjid_base", how="left", validate="many_to_one")


def load_subject_info(dataset_name: str) -> pd.DataFrame:
    if dataset_name not in ALLOWED_DATASETS:
        raise ValueError(f"Unknown dataset '{dataset_name}'. Allowed datasets: {ALLOWED_DATASETS}")

    info_path = DATASET_DIR / dataset_name / "subject_info.data"

    if not info_path.is_file():
        raise FileNotFoundError(f"Subject info file does not exist: {info_path}")

    df = pd.read_csv(info_path, sep=",")

    required_cols = ["subjid", "age", "sex"]
    missing_required = [col for col in required_cols if col not in df.columns]

    if missing_required:
        raise ValueError(f"{info_path} is missing required columns: {missing_required}")

    optional_drop_cols = [col for col in ("date_birth", "date_exam") if col in df.columns]

    if optional_drop_cols:
        df = df.drop(columns=optional_drop_cols)

    left_cols = ["subjid", "age", "sex"]
    other_cols = [col for col in df.columns if col not in left_cols]

    return df[left_cols + other_cols]



def load_aparcstats(dataset_name: str, results_dir: str | Path, measure: str = "volume") -> pd.DataFrame:

    if measure not in _APARC_MEASURES:
        raise ValueError(f"measure must be one of {sorted(_APARC_MEASURES)}.")

    stats_dir = Path(results_dir) / dataset_name / "stats"

    if not stats_dir.is_dir():
        raise NotADirectoryError(f"Structural stats directory does not exist: {stats_dir}")

    hemis = []

    for hemi in ("lh", "rh"):
        path = stats_dir / f"{hemi}.aparc.{measure}.stats"

        if not path.is_file():
            raise FileNotFoundError(f"Missing aparc stats file: {path}")

        df = pd.read_csv(path)

        if df.empty:
            raise ValueError(f"Aparc stats file is empty: {path}")

        df = df.rename(columns={df.columns[0]: "subjid"})
        df = add_base_subjid(df, source_col="subjid")

        duplicate_mask = df["subjid"].duplicated(keep=False)

        if duplicate_mask.any():
            duplicates = df.loc[duplicate_mask, "subjid"].tolist()
            raise ValueError(f"Duplicate subjects in {path}: {duplicates}")

        renamed_rois = {}

        for column in df.columns:
            if column in {"subjid", "subjid_base"}:
                continue

            roi = column.removeprefix(f"{hemi}_").removesuffix(f"_{measure}")
            renamed_rois[column] = f"ctx-{hemi}-{roi}"

        df = df.rename(columns=renamed_rois)

        roi_cols = [col for col in df.columns if col not in {"subjid", "subjid_base"}]
        df[roi_cols] = df[roi_cols].apply(pd.to_numeric, errors="raise")

        hemis.append(df)

    lh, rh = hemis

    lh_subjects = set(lh["subjid"])
    rh_subjects = set(rh["subjid"])

    only_lh = sorted(lh_subjects - rh_subjects)
    only_rh = sorted(rh_subjects - lh_subjects)

    if only_lh or only_rh:
        raise ValueError(
            "Left/right aparc subject mismatch.\n"
            f"Only in LH: {only_lh}\n"
            f"Only in RH: {only_rh}"
        )

    return lh.merge(rh, on=["subjid", "subjid_base"], how="inner", validate="one_to_one")



# Main

In [ ]:
DATASET_DIR = _require_env_path("DATASET_DIR")
RESULTS_DIR = _require_env_path("RESULTS_DIR_LARGE")
RESULTS_ROOT = RESULTS_DIR

files_agg = {
    "org": get_stats_table_files("/projects/b32903/Alex/results/STU00225089"),
    "synthsr": get_stats_table_files("/projects/b32903/Alex/results/STU00225089_synthsr"),
}

# CGI scores
df_cog = load_cogscores_batch(
    "/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx"
)

# For now df_model is simply the cognitive-score dataframe.
# No stimulus-data restriction is applied.
df_model = df_cog.copy()

# Load original + SynthSR modeled E-field subjects.
df_model_roi = build_model_roi(
    df_model=df_model,
    files_agg=files_agg,
    skin_model="skin_single",
    model_type="static",
)

# Add age / sex independently of df_stim and df_cog coverage.
df_subject_info = load_subject_info("STU00225089")
df_subject_info = add_base_subjid(df_subject_info, source_col="subjid")
df_model_roi    = add_subject_info(df_model_roi, df_subject_info)


# ============================================================================
# CHECKS
# ============================================================================

print("\nRows by modeling dataset:")
print(df_model_roi["is_synth"].value_counts(dropna=False))

print("\nUnique modeled subjects by modeling dataset:")
print(df_model_roi.groupby("is_synth")["subjid"].nunique())

print("\nUnique clinical subjects by modeling dataset:")
print(df_model_roi.groupby("is_synth")["subjid_base"].nunique())

# print("\nModeled subjects without cognitive-score rows:")
# print(df_model_roi.loc[df_model_roi["subjid_cog"].isna(), ["subjid_base"]].drop_duplicates().values)

# print("\nDataframe shape:")
# print(df_model_roi.shape)
# df_model_roi.to_csv('data_model_roi.csv')

# ============================================================================
# OPTIONAL STIMULUS DATA
#
# Deliberately disabled for now.
# Use later when a specific analysis requires stimulus/session data.
# ============================================================================

# df_stim = load_stimulus(
#     Path("/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/qual_results_june06_2026.xlsx")
# )
#
# df_stim = df_compute_ef_sf(df_stim)
#
# df_model_stim = df_combine_stim_cog(
#     df_stim,
#     df_cog,
#     output_path="data_model.csv",
# )


# ============================================================================
# OPTIONAL STRUCTURAL DATA
# ============================================================================

df_aparc_volume = load_aparcstats(
    dataset_name="STU00225089",
    results_dir=RESULTS_ROOT,
    measure="volume",
)

# df_aparc_thickness = load_aparcstats(
#     dataset_name="STU00225089",
#     results_dir=RESULTS_ROOT,
#     measure="thickness",
# )

# df_aparc_meancurv = load_aparcstats(
#     dataset_name="STU00225089",
#     results_dir=RESULTS_ROOT,
#     measure="meancurv",
# )

In [ ]:
df_aparc_volume

In [ ]:
def load_stimulus_new(fn:str=None, sheet_name:str = None):
    if not fn: 
        raise ValueError('filename (fn) must be defined')
    
    if not isinstance(sheet_name, str):
        return pd.read_excel(fn)
    
    return pd.read_excel(fn, sheet_name=sheet_name)

fn_stim = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx'
df_stim = load_stimulus_new(fn_stim, sheet_name='stimulus')
df_stim = df_stim[df_stim.series_num != 0] # 0 = maintenance
df_stim = df_stim[df_stim.frequency_hz != 'age'] # 0 = maintenance
df_stim['ef_sf'] = df_stim["pulse_width_ms"] * df_stim["frequency_hz"] * df_stim["percent_charge"]
df_stim

## CGI Summary 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class FigCGIViolinStartEnd:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def run(self):
        tmp = self.df[['subjid', 'cgi_start', 'cgi_end', 'sex']].copy()
        tmp['sex'] = tmp['sex'].replace({'M': 'Male', 'F': 'Female'})
        tmp[['cgi_start', 'cgi_end']] = tmp[['cgi_start', 'cgi_end']].apply(pd.to_numeric, errors='coerce')
        
        tmp = tmp.dropna(subset=['cgi_start', 'cgi_end']).reset_index(drop=True)

        if len(tmp) < 2:
            print('Need at least 2 complete start/end pairs.')
            return

        start = tmp['cgi_start'].to_numpy()
        end = tmp['cgi_end'].to_numpy()

        start_mean, start_sd = start.mean(), start.std(ddof=1)
        end_mean, end_sd = end.mean(), end.std(ddof=1)
        global_mean = np.concatenate([start, end]).mean()

        fig, ax = plt.subplots(figsize=(8, 6))

        parts = ax.violinplot([start, end], positions=[1, 2], showmeans=True, showmedians=True, showextrema=True)

        violin_colors = ['tab:blue', 'orange']
        for body, color in zip(parts['bodies'], violin_colors):
            body.set_facecolor(color)
            body.set_edgecolor('black')
            body.set_alpha(0.5)

        for key in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
            if key in parts:
                parts[key].set_color('black')
                parts[key].set_linewidth(1.2)

        rng = np.random.default_rng(42)
        jitter = rng.uniform(-0.06, 0.06, size=len(tmp))
        x_start, x_end = 1 + jitter, 2 + jitter

        for i in range(len(tmp)):
            ax.plot([x_start[i], x_end[i]], [start[i], end[i]], color='gray', alpha=0.45, linewidth=1)

        for sex, marker, size in [('Male', 'o', 35), ('Female', '*', 80)]:
            mask = tmp['sex'].eq(sex)
            ax.scatter(x_start[mask], start[mask], s=size, marker=marker, alpha=0.8, color='tab:blue', edgecolors='black', linewidths=0.4, label=f'Start — {sex}')
            ax.scatter(x_end[mask], end[mask], s=size, marker=marker, alpha=0.8, color='orange', edgecolors='black', linewidths=0.4, label=f'End — {sex}')

        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Start\n{start_mean:.1f} ± {start_sd:.1f}', f'End\n{end_mean:.1f} ± {end_sd:.1f}'])
        ax.set_ylabel('CGI')
        ax.set_ylim([1, 8])
        # ax.set_ybound([0, 8])
        ax.set_yticklabels([i for i in range(1,8)])
        ax.set_title('CGI at Start and End')
        ax.grid(True)
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(f'Start:       n={len(start)}, mean={start_mean:.2f}, SD={start_sd:.2f}, var={start.var(ddof=1):.2f}')
        print(f'End:         n={len(end)}, mean={end_mean:.2f}, SD={end_sd:.2f}, var={end.var(ddof=1):.2f}')
        print(f'Global mean: {global_mean:.2f}')

df = df_cog.copy()
df = df[df['batch_num'] != -1]

df = add_subject_info(df_cog, df_subject_info)

FigCGIViolinStartEnd(df.copy()).run()

del df, response_summary

## Next

In [ ]:
# ============================================================================
# STRUCTURAL DATASET CONFIG
# ============================================================================

RESULTS_ROOT = os.getenv("RESULTS_DIR_LARGE")

if not RESULTS_ROOT:
    raise EnvironmentError("Required environment variable 'RESULTS_DIR_LARGE' is not set.")

STRUCTURAL_DATASETS = {
    "org": {"dataset_name": "STU00225089", "is_synth": False},
    "synthsr": {"dataset_name": "STU00225089_synthsr", "is_synth": True},
}


def load_asegstats(dataset_name: str, results_dir: str | Path = RESULTS_ROOT) -> pd.DataFrame:
    results_dir = Path(results_dir)
    fn = results_dir / dataset_name / "stats" / "aseg.stats"

    if not fn.is_file():
        raise FileNotFoundError(f"aseg stats file does not exist: {fn}")

    df = pd.read_csv(fn)

    if "Measure:volume" in df.columns:
        df = df.rename(columns={"Measure:volume": "subjid"})
    elif "subjid" not in df.columns:
        raise KeyError(f"Could not find subject-ID column in {fn}. Expected 'Measure:volume' or 'subjid'.")

    df = add_base_subjid(df, source_col="subjid")

    duplicate_mask = df["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Duplicate modeled subject IDs found in {fn}: {duplicates}")

    df["dataset"] = dataset_name

    return df

def load_all_asegstats(results_dir: str | Path = RESULTS_ROOT) -> pd.DataFrame:
    dfs = []

    for source, config in STRUCTURAL_DATASETS.items():
        df = load_asegstats(config["dataset_name"], results_dir=results_dir)
        df["is_synth"] = config["is_synth"]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    duplicate_mask = df.duplicated(subset=["subjid", "is_synth"], keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, ["subjid", "is_synth"]]
        raise ValueError(f"Duplicate modeled subjects after combining datasets:\n{duplicates.to_string(index=False)}")

    return df

def _lmm_load_aparcaseg_structural(results_dir: str | Path = RESULTS_ROOT):
    dfs_volume = []
    dfs_thickness = []
    dfs_meancurv = []

    for source, config in STRUCTURAL_DATASETS.items():
        dataset_name = config["dataset_name"]
        is_synth = config["is_synth"]

        df_volume = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="volume")
        df_thickness = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="thickness")
        df_meancurv = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="meancurv")

        if "subjid_base" not in df_volume.columns:
            df_volume = add_base_subjid(df_volume, source_col="subjid")

        if "subjid_base" not in df_thickness.columns:
            df_thickness = add_base_subjid(df_thickness, source_col="subjid")

        if "subjid_base" not in df_meancurv.columns:
            df_meancurv = add_base_subjid(df_meancurv, source_col="subjid")

        df_volume["is_synth"] = is_synth
        df_thickness["is_synth"] = is_synth
        df_meancurv["is_synth"] = is_synth

        dfs_volume.append(df_volume)
        dfs_thickness.append(df_thickness)
        dfs_meancurv.append(df_meancurv)

    df_volume = pd.concat(dfs_volume, ignore_index=True)
    df_thickness = pd.concat(dfs_thickness, ignore_index=True)
    df_meancurv = pd.concat(dfs_meancurv, ignore_index=True)

    keys = ["subjid", "is_synth"]

    for name, df in {"volume": df_volume, "thickness": df_thickness, "meancurv": df_meancurv}.items():
        duplicate_mask = df.duplicated(subset=keys, keep=False)

        if duplicate_mask.any():
            duplicates = df.loc[duplicate_mask, keys]
            raise ValueError(f"Duplicate modeled subjects in {name} data:\n{duplicates.to_string(index=False)}")

    return df_volume, df_thickness, df_meancurv

def load_subject_info(dataset_name: str) -> pd.DataFrame:
    if dataset_name != "STU00225089":
        raise ValueError(f"Subject info should be loaded from STU00225089, not '{dataset_name}'.")

    root = os.getenv("DS_NMHC_ROOT")

    if not root:
        raise EnvironmentError("Required environment variable 'DS_NMHC_ROOT' is not set.")

    info_path = Path(root) / "subject_info.data"

    if not info_path.is_file():
        raise FileNotFoundError(f"Subject info file does not exist: {info_path}")

    df = pd.read_csv(info_path, sep=",")

    required_cols = ["subjid", "age", "sex"]
    missing_required = [col for col in required_cols if col not in df.columns]

    if missing_required:
        raise ValueError(f"{info_path} is missing required columns: {missing_required}")

    optional_drop_cols = [col for col in ("date_birth", "date_exam") if col in df.columns]

    if optional_drop_cols:
        df = df.drop(columns=optional_drop_cols)

    df = add_base_subjid(df, source_col="subjid")
    df["age"] = pd.to_numeric(df["age"], errors="raise")

    if df[["subjid", "subjid_base", "age", "sex"]].isna().any().any():
        bad_rows = df.index[df[["subjid", "subjid_base", "age", "sex"]].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing subject-info values at rows: {bad_rows}")

    exact_conflicts = df.groupby("subjid")[["age", "sex"]].nunique(dropna=False).gt(1)

    if exact_conflicts.any().any():
        bad_subjects = exact_conflicts.index[exact_conflicts.any(axis=1)].tolist()
        raise ValueError(f"Conflicting subject-info values for exact subject IDs: {bad_subjects}")

    left_cols = ["subjid", "subjid_base", "age", "sex"]
    other_cols = [col for col in df.columns if col not in left_cols]

    return df[left_cols + other_cols]

def df_combine_aseg2info(df_aseg: pd.DataFrame, df_info: pd.DataFrame) -> pd.DataFrame:
    """
    Attach age/sex to modeled scans.

    Matching priority:
        1. Exact modeled ID: subj-cat-067-002 -> subj-cat-067-002
        2. Base subject fallback: subj-cat-067-002 -> subj-cat-067

    Base-ID fallback is only allowed when age and sex are unambiguous.
    """

    df_aseg = df_aseg.copy()
    df_info = df_info.copy()

    required_aseg = {"subjid", "subjid_base", "is_synth"}
    required_info = {"subjid", "subjid_base", "age", "sex"}

    missing_aseg = required_aseg - set(df_aseg.columns)
    missing_info = required_info - set(df_info.columns)

    if missing_aseg:
        raise KeyError(f"df_aseg missing required columns: {sorted(missing_aseg)}")

    if missing_info:
        raise KeyError(f"df_info missing required columns: {sorted(missing_info)}")

    df_info["age"] = pd.to_numeric(df_info["age"], errors="raise")

    if df_info[["subjid", "subjid_base", "age", "sex"]].isna().any().any():
        bad_rows = df_info.index[df_info[["subjid", "subjid_base", "age", "sex"]].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing subject-info values at rows: {bad_rows}")

    # ------------------------------------------------------------------------
    # Exact modeled-ID subject information
    # ------------------------------------------------------------------------

    exact_conflicts = df_info.groupby("subjid")[["age", "sex"]].nunique(dropna=False).gt(1)

    if exact_conflicts.any().any():
        bad_subjects = exact_conflicts.index[exact_conflicts.any(axis=1)].tolist()
        raise ValueError(f"Conflicting subject-info values for exact modeled IDs: {bad_subjects}")

    df_info_exact = df_info.groupby("subjid", as_index=False).agg(
        subjid_base_info=("subjid_base", "first"),
        age=("age", "first"),
        sex=("sex", "first"),
    )

    result = df_aseg.merge(df_info_exact, on="subjid", how="left", validate="many_to_one")

    # Exact ID must also map to the expected base subject.
    mismatch = result["subjid_base_info"].notna() & result["subjid_base_info"].ne(result["subjid_base"])

    if mismatch.any():
        bad = result.loc[mismatch, ["subjid", "subjid_base", "subjid_base_info"]]
        raise ValueError(f"Exact-ID/base-ID mismatch:\n{bad.to_string(index=False)}")

    result = result.drop(columns=["subjid_base_info"])

    # ------------------------------------------------------------------------
    # Base-ID fallback for modeled scans without exact metadata
    # ------------------------------------------------------------------------

    missing_exact = result["age"].isna() | result["sex"].isna()

    if missing_exact.any():
        fallback_subjects = set(result.loc[missing_exact, "subjid_base"])
        df_info_fallback = df_info[df_info["subjid_base"].isin(fallback_subjects)].copy()

        available_fallback_subjects = set(df_info_fallback["subjid_base"])
        no_fallback = sorted(fallback_subjects - available_fallback_subjects)

        if no_fallback:
            raise ValueError(f"No subject-info data exists for base subjects: {no_fallback}")

        age_conflicts = df_info_fallback.groupby("subjid_base")["age"].nunique(dropna=False).gt(1)
        sex_conflicts = df_info_fallback.groupby("subjid_base")["sex"].nunique(dropna=False).gt(1)

        if age_conflicts.any():
            subjects = age_conflicts[age_conflicts].index.tolist()
            raise ValueError(
                "Cannot use base-subject fallback because multiple ages exist and scan chronology is unknown: "
                f"{subjects}"
            )

        if sex_conflicts.any():
            subjects = sex_conflicts[sex_conflicts].index.tolist()
            raise ValueError(f"Cannot use base-subject fallback because conflicting sex values exist: {subjects}")

        df_info_fallback = df_info_fallback.groupby("subjid_base", as_index=False).agg(
            age_fallback=("age", "first"),
            sex_fallback=("sex", "first"),
        )

        result = result.merge(df_info_fallback, on="subjid_base", how="left", validate="many_to_one")
        result["age"] = result["age"].fillna(result["age_fallback"])
        result["sex"] = result["sex"].fillna(result["sex_fallback"])
        result = result.drop(columns=["age_fallback", "sex_fallback"])

    # ------------------------------------------------------------------------
    # Final validation
    # ------------------------------------------------------------------------

    if result[["age", "sex"]].isna().any().any():
        bad = result.loc[result[["age", "sex"]].isna().any(axis=1), ["subjid", "subjid_base", "is_synth"]]
        raise ValueError(f"No subject-info data could be assigned to modeled subjects:\n{bad.to_string(index=False)}")

    if len(result) != len(df_aseg):
        raise RuntimeError(f"Subject-info merge changed row count: {len(df_aseg)} -> {len(result)}")

    left_cols = ["subjid", "subjid_base", "is_synth", "age", "sex"]
    other_cols = [col for col in result.columns if col not in left_cols]

    return result[left_cols + other_cols].sort_values(["is_synth", "subjid"]).reset_index(drop=True)

df_aseg         = load_all_asegstats()
df_subject_info = load_subject_info("STU00225089")
df_aseg         = df_combine_aseg2info(df_aseg, df_subject_info)

df_volume, df_thickness, df_meancurv = _lmm_load_aparcaseg_structural()

fn = 'data_model_roi.csv'
df_stats_field_aparcseg = load_aparcaseg_table(fn)

cols_exclude_tmp = {"subjid", "subjid_base", "Unknown", "# Unknown"}
cols_aparcseg = [col for col in df_stats_field_aparcseg.columns if col not in cols_exclude_tmp]

required_aseg_etiv = ["subjid", "subjid_base", "is_synth", "EstimatedTotalIntraCranialVol"]
required_aparc_etiv = ["subjid", "subjid_base", "is_synth", "ctx-rh-eTIV", "ctx-lh-eTIV"]

missing_aseg_etiv = [col for col in required_aseg_etiv if col not in df_aseg.columns]
missing_aparc_etiv = [col for col in required_aparc_etiv if col not in df_volume.columns]

if missing_aseg_etiv:
    raise KeyError(f"df_aseg missing ETIV columns: {missing_aseg_etiv}")

if missing_aparc_etiv:
    raise KeyError(f"df_volume missing ETIV columns: {missing_aparc_etiv}")

df_aseg_etiv = df_aseg[required_aseg_etiv].copy()
df_aparc_etiv = df_volume[required_aparc_etiv].copy()

df_aparc_etiv["ctx-eTIV"] = df_aparc_etiv["ctx-rh-eTIV"] + df_aparc_etiv["ctx-lh-eTIV"]

merge_keys = ["subjid", "subjid_base", "is_synth"]

duplicate_aseg = df_aseg_etiv.duplicated(subset=merge_keys, keep=False)

if duplicate_aseg.any():
    duplicates = df_aseg_etiv.loc[duplicate_aseg, merge_keys]
    raise ValueError(f"Duplicate modeled subjects in aseg ETIV data:\n{duplicates.to_string(index=False)}")

duplicate_aparc = df_aparc_etiv.duplicated(subset=merge_keys, keep=False)

if duplicate_aparc.any():
    duplicates = df_aparc_etiv.loc[duplicate_aparc, merge_keys]
    raise ValueError(f"Duplicate modeled subjects in aparc ETIV data:\n{duplicates.to_string(index=False)}")

aseg_keys = set(map(tuple, df_aseg_etiv[merge_keys].to_numpy()))
aparc_keys = set(map(tuple, df_aparc_etiv[merge_keys].to_numpy()))

only_aseg = sorted(aseg_keys - aparc_keys)
only_aparc = sorted(aparc_keys - aseg_keys)

if only_aseg or only_aparc:
    raise ValueError(
        "Modeled-subject mismatch between aseg and aparc ETIV data.\n"
        f"Only in aseg ({len(only_aseg)}): {only_aseg}\n"
        f"Only in aparc ({len(only_aparc)}): {only_aparc}"
    )

df_etiv = df_aseg_etiv.merge(df_aparc_etiv, on=merge_keys, how="inner", validate="one_to_one")


# ============================================================================
# CHECKS
# ============================================================================

print("\nASEG modeled subjects:")
print(df_aseg.groupby("is_synth")["subjid"].nunique())

print("\nAPARC modeled subjects:")
print(df_volume.groupby("is_synth")["subjid"].nunique())

print("\nETIV modeled subjects:")
print(df_etiv.groupby("is_synth")["subjid"].nunique())

print("\nUnique base subjects:")
print(df_etiv.groupby("is_synth")["subjid_base"].nunique())

print("\nETIV dataframe shape:")
print(df_etiv.shape)

# Infer Electrode placement (will be standalone)

In [ ]:
fn_stim = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx'
df_stim = load_stimulus_new(fn_stim, sheet_name='stimulus')

df_elec = df_stim[['subjid','electrode_placement']]

df_elec = df_elec.drop_duplicates()
df_elec = df_elec.dropna(axis=0)
df_elec

# October 14, 2026

## Prep

In [13]:
import pandas as pd
import numpy as np 
from tabulate import tabulate
from simnibs_parcel_analysis import summarize_treatment_courses

fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest.xlsx'
fn_query = '/projects/p32903/Alex2/ect-results-tools/query_result_1.csv'

def _add_person_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_id"] = df["subjid"].str.extract(
        r"^(subj-cat-\d{3})(?:-\d{3})?$", expand=False
    )

    invalid = df["person_id"].isna()
    if invalid.any():
        raise ValueError(
            f"Invalid/missing subjid at rows {df.index[invalid].tolist()}"
        )
    return df

def load_stimulation_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="stimulus")
    return _add_person_id(df)

def load_course_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="Sheet1")
    df = df.loc[
        df["in_stim"].eq(True) & df["batch_num"].gt(0)
    ].copy()

    df["cgi_change"] = df["cgi_start"] - df["cgi_end"]
    return _add_person_id(df)

def load_query_results(fn: str) -> pd.DataFrame:
    fs_subjects_dirs = {
        "STU00225089": "/projects/p32903/Alex2/datasets/STU00225089/subjectsdir",
        "STU00225089_synthsr": "/projects/p32903/Alex2/datasets/STU00225089/synthsr/subjectsdir"
    }
    df_query = pd.read_csv(fn)
    df_query["fs_subjects_dir"] = df_query["dataset_root"].map(fs_subjects_dirs)

    if df_query["fs_subjects_dir"].isna().any():
        missing = df_query.loc[df_query["fs_subjects_dir"].isna(), "dataset_root"].unique()
        raise ValueError(f"Missing FreeSurfer SUBJECTS_DIR mapping for: {missing}")
    return df_query

def _compute_ef_sf(df:pd.DataFrame)->pd.DataFrame:
    cols    = ['pulse_width_ms', 'frequency_hz', 'percent_charge']
    df      = df.copy()
    df['ef_sf'] = df['pulse_width_ms'] * df['frequency_hz'] * df['percent_charge']
    return df


## 1) LMM

### 1.1) `Tx ~ 1 + age*sex + electrode_placement`


Using second session parameters to summarize treatment course

In [ ]:
def run_null(df:pd.DataFrame):
    import statsmodels.formula.api as smf
    df = _prepare_df_null()
    formula = "cgi_change ~ 1 + age * C(sex) + C(electrode_placement)"
    return smf.ols(formula, data=df, missing="raise").fit()

### COLLAPSE SESSION -> COURSE BY SECOND SESSION ###
def _prepare_df_null()->pd.DataFrame:
    fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest.xlsx'

    df_cgi  = load_course_data(fn_cgi)
    df_stim = load_stimulation_data(fn_cgi)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables
    
    df_stim = df_stim[df_stim['session_num']==2]
    df_stim.drop(columns=['session_num'])

    # cgi_cols_keep   = ['subjid', 'sex', 'age', 'cgi_start','series_num', 'cgi_end', 'cgi_change', 'response_type']
    # df_cgi_filtered = df_cgi[cgi_cols_keep].copy()

    shared_metadata = [
    col for col in ["age", "sex", "dob"]
        if col in df_stim.columns and col in df_cgi.columns
    ]

    df_combined = df_stim.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    return df_combined

df = _prepare_df_null()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
result = run_null(df)
result.summary()

### 1.2) `Tx ~ 1 + E_x + E_x:ef_sf + (1|age) + (1|sex)`

### 1.3) `Tx ~ 1 + E_global_mean + E_global_mean:ef_sf + (1|age) + (1|sex)`

In [ ]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
import faulthandler
fault_log = open("kernel_fault.log", "w")
faulthandler.enable(file=fault_log, all_threads=True)

def _prepare_df_lmm_2a():
    df_query = load_query_results(fn_query)
    df_query = df_query[df_query['threshold']==95]
    df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
    query_filters = {"model_type_dir": "skin_double", 
                    "step_result": "fsavg_overlays",
                    "step": "step_4", 
                    "asf": "asf_5",
                    "threshold":95}

    df_cgi  = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables

    df_stim_mapped = map_session_files(
        sessions            = df_stim, 
        query               = df_query,
        filename            = "tdcs_uq_gpc_fsavg.msh",
        query_filters       = query_filters,
        subject_scan_map    ="subject_scan_map.json",
    )
    df_stim_mapped = df_stim_mapped[df_stim_mapped['session_num']==2]

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_stim_mapped,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=95,
        output_format="wide",
    )

    df_E_global = global_mean_e(
        (simnibs.read_msh(path) for path in df_mesh_roi["full_file_path"]),
        field_name="magnE_mean",
    )

    df_E_global = pd.DataFrame({
        "full_file_path": df_mesh_roi["full_file_path"].to_numpy(),
        "e_global": df_E_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    df_stim_global.index = df_stim_mapped.index.copy()
    shared_metadata = [
        col for col in ["age", "sex", "dob"]
            if col in df_stim_global.columns and col in df_cgi.columns
    ]
    
    df_combined = df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    df_combined = df_combined.rename(columns={"e_global": "E_global_mean"})
    return df_combined

def run_2a(df:pd.DataFrame):
    import numpy as np
    import statsmodels.formula.api as smf

    cols = ["cgi_change", "E_global_mean", "ef_sf", "age", "sex"]
    df = df[cols].copy()

    if df.isna().any().any():
        raise ValueError("Missing values in model columns.")

    df["sex"] = df["sex"].astype("category")

    model = smf.mixedlm(
        "cgi_change ~ 1 + E_global_mean + E_global_mean:ef_sf",
        data=df,
        groups=np.ones(len(df)),
        re_formula="0",
        vc_formula={
            "age": "0 + C(age)",
            "sex": "0 + C(sex)",
        },
    )

    result = model.fit(reml=True, method="lbfgs")
    
    print(result.summary())
    return result

df = _prepare_df_lmm_2a()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
# result = run_2a(df)

### 1.4) `Tx ~ 1 + E_95_parcel + E_95_parcel:ef_sf + (1|age) + (1|sex)`

In [ ]:
"""
1. load fsaverage meshes
2. get all ROIs
"""

from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e

df_query = load_query_results(fn_query)
df_query = df_query[df_query['threshold']==95]
df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
query_filters = {"model_type_dir": "skin_double", 
                 "step_result": "fsavg_overlays",
                 "step": "step_4", 
                 "asf": "asf_5",
                 "threshold":95}

df_cgi  = load_course_data(fn_cgi)

df_stim = load_stimulation_data(fn_cgi)
df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']

df_stim_mapped = map_session_files(
    sessions            = df_stim, 
    query               = df_query,
    filename            = "tdcs_uq_gpc_fsavg.msh",
    query_filters       = query_filters,
    subject_scan_map    ="subject_scan_map.json",
)

df_mesh_roi = extract_fsaverage_roi_metrics(
    df_stim_mapped,
    path_column="full_file_path",
    field_name="magnE_mean",
    atlas_name="HCP_MMP1",
    min_nodes_per_roi=20,
    percentile=95,
    output_format="wide",
)

# # keep percentile | mean columns only, drop other
# df_mean = df_mesh_roi.loc[
#     :, ~df_mesh_roi.columns.str.endswith("_e_percentile")
# ].copy()

# df_percentile = df_mesh_roi.loc[
#     :, ~df_mesh_roi.columns.str.endswith("_e_mean")
# ].copy()

# # Attach parcel metrics to each original session.
# df_stim_roi = join_session_roi_metrics(
#     df_stim_mapped,
#     df_mean,
#     path_column="full_file_path",
# )

# df_stim_roi = df_stim_roi[df_stim_roi['session_num']==2]
# df_stim_roi

## 2) Correlations

### Functions

In [14]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from itertools import product
import numpy as np
import pandas as pd
from typing import Literal
from scipy.stats import pearsonr, spearmanr

def _prepare_df_efield(threshold: int, step: str, model_type: str) -> pd.DataFrame:

    df_query = load_query_results(fn_query)
    df_query = df_query[df_query["subject_dir"] != "subj-cat-010-001"]

    query_filters = {
        "model_type_dir": model_type,
        "step_result": "fsavg_overlays",
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim["subjid"] != "subj-cat-010-001"]
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    df_stim_mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc_fsavg.msh",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    # Temporary course-level representation.
    df_stim_mapped = df_stim_mapped.loc[df_stim_mapped["session_num"] == 2].copy()

    if df_stim_mapped.empty:
        raise ValueError(f"No eligible sessions for {model_type}, {step}, threshold={threshold}")

    if df_stim_mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths.")

    # Compute global E-field only once per unique mesh.
    paths = df_stim_mapped["full_file_path"].drop_duplicates().to_numpy()

    e_global = global_mean_e(
        (simnibs.read_msh(path) for path in paths),
        field_name="magnE_mean",
    )

    if e_global.shape[0] != len(paths):
        raise ValueError("E-field result count does not match the number of unique meshes.")

    df_E_global = pd.DataFrame({
        "full_file_path": paths,
        "E_global_mean": e_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    shared_metadata = [
        col for col in ["age", "sex", "dob"]
        if col in df_stim_global.columns and col in df_cgi.columns
    ]

    return df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )

def run_efield_correlation(
    df: pd.DataFrame,
    x: str = "E_global_mean",
    y: str = "cgi_change",
    method: Literal["pearson", "spearman"] = "pearson") -> dict:
    
    methods = {"pearson": pearsonr, "spearman": spearmanr}
    if method not in methods:
        raise ValueError(f"Invalid method {method!r}. Expected one of {list(methods)}")

    required = ["person_id", "series_num", x, y]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    if df.duplicated(["person_id", "series_num"]).any():
        raise ValueError("Duplicate treatment courses detected.")

    pairs = df[[x, y]].apply(pd.to_numeric, errors="raise")
    valid = pairs.notna().all(axis=1)
    pairs = pairs.loc[valid]

    if not np.isfinite(pairs.to_numpy()).all():
        raise ValueError("Non-finite correlation values detected.")

    result = {
        "method": method,
        "x": x,
        "y": y,
        "n_courses": len(pairs),
        "n_patients": df.loc[valid, "person_id"].nunique(),
        "n_missing": len(df) - len(pairs),
        "r": np.nan,
        "p": np.nan,
        "status": "ok",
    }

    if len(pairs) < 3:
        result["status"] = "insufficient_n"
    elif (pairs.nunique() < 2).any():
        result["status"] = "constant_variable"
    else:
        corr = methods[method](pairs[x], pairs[y])
        result.update(r=float(corr.statistic), p=float(corr.pvalue))

    return result


### 2.1) Tx ~ E_global_mean

In [ ]:
from pathlib import Path

df_query = load_query_results(fn_query)

steps       = sorted(df_query["step"].dropna().unique())
model_types = sorted(df_query["model_type_dir"].dropna().unique())
thresholds  = sorted(df_query["threshold"].dropna().unique())

from itertools import product
from tqdm import tqdm
from itertools import product
import time

configs = list(product(model_types, steps, thresholds))
print(f"total combinations for config: {len(configs)}")
methods = ["pearson", "spearman"]
results = []

with tqdm( total=len(configs) * len(methods), desc="E-field correlations",
          unit="corr", mininterval=0, miniters=1) as pbar:
    
    for model_type, step, threshold in configs:
        pbar.set_postfix(model=model_type, step=step, threshold=threshold, refresh=True)
        start = time.perf_counter()

        df = _prepare_df_efield(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
        )

        for method in methods:
            corr = run_efield_correlation(df, method=method)

            results.append({
                "model_type": model_type,
                "step": step,
                "threshold": threshold,
                "percentile": percentile,
                **corr,
            })

            pbar.update(1)
            pbar.refresh()

        pbar.set_postfix(last_sec=round(time.perf_counter() - start, 1), refresh=True)

df_results = pd.DataFrame(results)

fn_out = Path("results/correlations/global_efield.csv")
fn_out.parent.mkdir(parents=True, exist_ok=True)

df_results.to_csv(fn_out, index=False)

### 2.2) Tx ~ E_parcel_95 

In [18]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from itertools import product
import numpy as np
import pandas as pd
from typing import Literal
from scipy.stats import pearsonr, spearmanr
from pathlib import Path
from itertools import product
from tqdm.auto import tqdm

def _prepare_df_efield_roi(
    threshold: int,
    step: str,
    model_type: str,
    percentile: float = 95,
    combine_hemis: bool = False) -> pd.DataFrame:

    df_query = load_query_results(fn_query)
    df_query = df_query.loc[df_query["subject_dir"] != "subj-cat-010-001"].copy()

    query_filters = {
        "model_type_dir": model_type,
        "step_result": "fsavg_overlays",
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim.loc[df_stim["subjid"] != "subj-cat-010-001"].copy()
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    df_stim_mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc_fsavg.msh",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    # Temporary course-level representation.
    df_stim_mapped = df_stim_mapped.loc[df_stim_mapped["session_num"] == 2].copy()

    if df_stim_mapped.empty:
        raise ValueError(f"No eligible sessions for {model_type}, {step}, threshold={threshold}")

    if df_stim_mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths.")

    # Extract ROI metrics once per unique mesh.
    df_unique = df_stim_mapped.drop_duplicates(subset=["full_file_path"]).copy()

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_unique,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=percentile,
        output_format="wide",
    )

    # Identify hemisphere-specific ROI columns.
    roi_cols = [
        col for col in df_mesh_roi.columns
        if isinstance(col, str) and col.startswith(("lh.", "rh."))
    ]

    if not roi_cols:
        raise ValueError("No lh./rh. ROI columns found in extracted metrics.")

    df_roi = df_mesh_roi[["full_file_path", *roi_cols]].copy()

    if df_roi["full_file_path"].isna().any() or df_roi["full_file_path"].duplicated().any():
        raise ValueError("ROI metrics must contain one row per unique, non-missing mesh path.")

    # Optionally combine corresponding left/right ROIs.
    if combine_hemis:
        lh = {col[3:]: col for col in roi_cols if col.startswith("lh.")}
        rh = {col[3:]: col for col in roi_cols if col.startswith("rh.")}

        if set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROIs: {sorted(set(lh) ^ set(rh))}")

        bilateral = pd.DataFrame({
            roi: (df_roi[lh[roi]] + df_roi[rh[roi]]) / 2
            for roi in sorted(lh)
        }, index=df_roi.index)

        df_roi = pd.concat([df_roi[["full_file_path"]], bilateral], axis=1)

    # Merge ROI metrics back to treatment sessions.
    overlap = (set(df_roi.columns) & set(df_stim_mapped.columns)) - {"full_file_path"}
    if overlap:
        raise ValueError(f"ROI columns conflict with session columns: {sorted(overlap)}")

    df_stim_roi = df_stim_mapped.merge(
        df_roi,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    shared_metadata = [
        col for col in ["age", "sex", "dob"]
        if col in df_stim_roi.columns and col in df_cgi.columns
    ]

    return df_stim_roi.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )

from typing import Literal
import pandas as pd


def run_roi_correlations(
    df: pd.DataFrame,
    combine_hemis: bool = False,
    metric: Literal["mean", "percentile"] = "percentile",
    method: Literal["pearson", "spearman"] = "pearson",
) -> pd.DataFrame:

    if metric not in ("mean", "percentile"):
        raise ValueError(f"Invalid metric: {metric!r}")

    suffix = f"_e_{metric}"

    roi_cols = [
        col for col in df.columns
        if isinstance(col, str)
        and col.endswith(suffix)
        and (not col.startswith(("lh.", "rh.")) if combine_hemis else col.startswith(("lh.", "rh.")))
        and "???" not in col
    ]

    if not roi_cols:
        raise ValueError(f"No ROI columns found for combine_hemis={combine_hemis}, metric={metric}")

    results = []

    for col in roi_cols:
        corr = run_efield_correlation(df, x=col, y="cgi_change", method=method)

        hemisphere = col.split(".", 1)[0] if not combine_hemis else "bilateral"
        roi = col.split(".", 1)[-1].removesuffix(suffix)

        results.append({
            "roi": roi,
            "hemisphere": hemisphere,
            "roi_column": col,
            "metric": metric,
            "combine_hemis": combine_hemis,
            **corr,
        })

    return pd.DataFrame(results)

In [ ]:
from itertools import product
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import time

df_query = load_query_results(fn_query)

# steps       = sorted(df_query["step"].dropna().unique())
model_types = sorted(df_query["model_type_dir"].dropna().unique())
thresholds  = sorted(df_query["threshold"].dropna().unique())
steps       = ['step_0','step_2', 'step_4']

percentiles = [95]  # supports multiple values, e.g. [90, 95, 99].
methods = ["pearson", "spearman"]
combine_hemis_options = [True, False]

if not percentiles or len(percentiles) != len(set(percentiles)):
    raise ValueError("percentiles must be nonempty and contain no duplicate values.")

configs = list(product(model_types, steps, thresholds, percentiles, combine_hemis_options))
results = []

# Per simulation/hemisphere configuration:
# one mean analysis + one analysis for each requested percentile, for each method.
n_base_configs = len(model_types) * len(steps) * len(thresholds) * len(combine_hemis_options)
total_batches = n_base_configs * (1 + len(percentiles)) * len(methods)

if not configs:
    raise ValueError("No simulation configurations to process.")

print(f"Total preparation configurations: {len(configs)}")
print(f"Total correlation batches: {total_batches}")

with tqdm(total=total_batches, desc="ROI correlations", unit="batch", mininterval=0, miniters=1) as pbar:
    for model_type, step, threshold, percentile, combine_hemis in configs:
        progress = {
            "model": model_type,
            "step": step,
            "threshold": threshold,
            "percentile": percentile,
            "hemis": combine_hemis,
        }

        pbar.set_postfix(**progress, status="preparing", refresh=True)
        start = time.perf_counter()

        # Both ROI means and the requested ROI percentiles are extracted here.
        df = _prepare_df_efield_roi(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
            percentile=percentile,
            combine_hemis=combine_hemis,
        )

        # Analyze means only during the first percentile pass to avoid duplicate tests.
        metrics = ["mean", "percentile"] if percentile == percentiles[0] else ["percentile"]

        for metric, method in product(metrics, methods):
            pbar.set_postfix(**progress, metric=metric, method=method, status="correlating", refresh=True)

            corr = run_roi_correlations(
                df,
                combine_hemis=combine_hemis,
                metric=metric,
                method=method,
            )

            corr["model_type"] = model_type
            corr["step"] = step
            corr["threshold"] = threshold
            corr["metric"] = metric

            # Percentile is an analysis parameter only for the percentile metric.
            corr["percentile"] = float(percentile) if metric == "percentile" else float("nan")

            results.append(corr)

            pbar.update(1)
            pbar.refresh()

        elapsed = round(time.perf_counter() - start, 1)
        pbar.set_postfix(**progress, status="done", last_sec=elapsed, refresh=True)

df_results = pd.concat(results, ignore_index=True)
df_results['r_abs'] = df_results[r]
output = Path("results/correlations/roi_efield.csv")
output.parent.mkdir(parents=True, exist_ok=True)
df_results.to_csv(output, index=False)

Save the results to .msh file

In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import re

def write_all_roi_correlation_meshes(
    results: pd.DataFrame,
    template_path: str | Path,
    output_dir: str | Path,
    *,
    atlas_name: str = "HCP_MMP1",
    value_columns: tuple[str, ...] = ("r", "r_abs", "p", "n_courses"),
    overwrite: bool = False,
) -> pd.DataFrame:
    """Export one mesh/source CSV per analysis present in results; return an export index."""
    if results.empty or not results.columns.is_unique:
        raise ValueError("Results must be nonempty and have unique column names.")
    if isinstance(value_columns, str) or not value_columns or len(set(value_columns)) != len(value_columns):
        raise ValueError("value_columns must be a nonempty sequence of unique column names.")

    config_cols = ["model_type", "step", "threshold", "percentile", "combine_hemis", "metric", "method", "y"]
    optional_config = [c for c in ("asf", "atlas", "field_name") if c in results.columns]
    group_cols = config_cols + optional_config

    required = {*group_cols, "roi", "hemisphere", "status"}
    missing = required - set(results.columns)
    if missing:
        raise ValueError(f"Missing result columns: {sorted(missing)}")

    frame = results.copy()

    # Derive absolute correlation only if requested and not already present.
    if "r_abs" in value_columns and "r_abs" not in frame.columns:
        if "r" not in frame.columns:
            raise ValueError("Cannot calculate r_abs without an r column.")
        frame["r_abs"] = pd.to_numeric(frame["r"], errors="raise").abs()

    missing = set(value_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing value columns: {sorted(missing)}")

    if not frame["combine_hemis"].map(lambda x: isinstance(x, (bool, np.bool_))).all():
        raise TypeError("combine_hemis must contain booleans, not strings or integers.")
    if frame[[c for c in group_cols if c != "percentile"]].isna().any().any():
        raise ValueError("Configuration values cannot be missing, except percentile for non-percentile metrics.")
    if "atlas" in frame.columns and not frame["atlas"].eq(atlas_name).all():
        raise ValueError("All results must match atlas_name; filter by atlas before exporting.")
    if frame.duplicated(group_cols + ["hemisphere", "roi"]).any():
        raise ValueError("Duplicate ROI results within an analysis; do not silently deduplicate them.")

    # Retain mean analyses, whose percentile values are NaN.
    grouped = frame.groupby(group_cols, dropna=False, observed=True, sort=True)

    output_dir = Path(output_dir).expanduser()
    index_path = output_dir / "export_index.csv"

    if index_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {index_path}")

    records = []
    used_paths = set()

    with tqdm(total=grouped.ngroups, desc="Exporting ROI maps", unit="mesh", mininterval=0, miniters=1) as pbar:
        for key, selected in grouped:
            config = dict(zip(group_cols, key, strict=True))
            hemi_label = "bilateral" if config["combine_hemis"] else "separate"

            metric_label = config["metric"]
            if metric_label == "percentile":
                percentile = float(config["percentile"])
                if not np.isfinite(percentile) or not 0 <= percentile <= 100:
                    raise ValueError(f"Invalid percentile in configuration: {config}")
                metric_label = f"p{percentile:g}"
            elif pd.notna(config["percentile"]):
                raise ValueError("Non-percentile metrics must have percentile=NaN.")

            # Include the outcome and optional settings to distinguish analyses.
            parts = [
                atlas_name, str(config["y"]), str(config["model_type"]), str(config["step"]),
                f"thr{float(config['threshold']):g}", metric_label, str(config["method"]), hemi_label,
            ]
            parts.extend(f"{c}-{config[c]}" for c in optional_config if c != "atlas")
            filename = re.sub(r"[^A-Za-z0-9_.-]+", "-", "_".join(parts)) + ".msh"
            output_path = output_dir / filename

            # Even with overwrite=True, different analyses must not share a filename.
            if output_path in used_paths:
                raise ValueError(f"Different analyses produced the same filename: {output_path}")
            used_paths.add(output_path)

            pbar.set_postfix(
                model=config["model_type"],
                step=config["step"],
                threshold=config["threshold"],
                metric=metric_label,
                method=config["method"],
                hemis=hemi_label,
                refresh=True,
            )

            output = write_roi_correlation_mesh(
                selected,
                template_path=template_path,
                output_path=output_path,
                atlas_name=atlas_name,
                value_columns=value_columns,
                overwrite=overwrite,
            )

            records.append({
                **config,
                "mesh_atlas": atlas_name,
                "n_roi_results": len(selected),
                "n_ok": int(selected["status"].eq("ok").sum()),
                "mesh_path": str(output.resolve()),
                "source_csv": str(output.with_suffix(".csv").resolve()),
            })

            pbar.update(1)
            pbar.refresh()

    export_index = pd.DataFrame(records)
    export_index.to_csv(index_path, index=False)

    return export_index


template_path = Path("/projects/p32903/Alex2/datasets/freesurfer_templates/tdcs_uq_gpc_fsavg.msh")
output_dir = Path("results/correlations/maps")

df_exports = write_all_roi_correlation_meshes(
    df_results,
    template_path=template_path,
    output_dir=output_dir,
    atlas_name="HCP_MMP1",
    value_columns=("r", "r_abs", "p", "n_courses"),
    overwrite=False,
)

print(f"Exported {len(df_exports)} meshes.")
print(f"Export index: {output_dir / 'export_index.csv'}")

display(df_exports[
    ["model_type", "step", "threshold", "metric", "percentile", "method",
     "combine_hemis", "n_roi_results", "n_ok", "mesh_path"]
])

### 2.3)  FreeSurfer Desikan–Killiany + aseg (aparc+aseg.mgz)

In [16]:
from numbers import Real
from typing import Literal

import numpy as np
import pandas as pd

from simnibs_parcel_analysis.atlas_volume import build_volume_roi_summary
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files

import importlib
import simnibs_parcel_analysis.atlas_volume as atlas_volume

importlib.reload(atlas_volume)
build_volume_roi_summary = atlas_volume.build_volume_roi_summary

def _volume_roi_name(name: str) -> str:
    """Normalize lateralized FreeSurfer labels to the existing lh./rh. ROI format."""
    prefixes = {
        "ctx-lh-": "lh.", "ctx-rh-": "rh.",
        "ctx_lh_": "lh.", "ctx_rh_": "rh.",
        "Left-": "lh.", "Right-": "rh.",
    }
    for prefix, replacement in prefixes.items():
        if name.startswith(prefix):
            return replacement + name[len(prefix):]
    raise ValueError(f"Unsupported or nonlateralized FreeSurfer ROI: {name!r}")


def _prepare_df_efield_volume(
    threshold: int,
    step: str,
    model_type: str,
    percentile: float = 95,
    combine_hemis: bool = False,
    percentile_weighting: Literal["unweighted", "spatial_weighted"] = "spatial_weighted",
    negative_atol: float = 1e-6,
) -> pd.DataFrame:
    """Prepare session-2 courses with aparc+aseg ROI means, percentiles, and CGI outcomes.

    Uses existing notebook globals: fn_query, fn_cgi, load_query_results,
    load_course_data, load_stimulation_data, and _compute_ef_sf.
    load_query_results must return dataset_root and fs_subjects_dir.
    Simulation paths must match df_query.full_file_path exactly.
    """
    if (isinstance(percentile, (bool, np.bool_)) or not isinstance(percentile, Real)
            or not np.isfinite(percentile) or not 0 <= percentile <= 100):
        raise ValueError("percentile must be a finite number in [0, 100].")
    if not isinstance(combine_hemis, (bool, np.bool_)):
        raise TypeError("combine_hemis must be a boolean.")
    if percentile_weighting not in {"unweighted", "spatial_weighted"}:
        raise ValueError("percentile_weighting must be 'unweighted' or 'spatial_weighted'.")

    # The updated loader supplies dataset_root -> fs_subjects_dir.
    df_query = load_query_results(fn_query)
    query_columns = ["full_file_path", "subject_dir", "dataset_root", "fs_subjects_dir"]
    missing = set(query_columns) - set(df_query.columns)
    if missing:
        raise ValueError(f"Missing query columns: {sorted(missing)}")
    df_query = df_query.loc[df_query["subject_dir"] != "subj-cat-010-001"].copy()

    query_filters = {
        "model_type_dir": model_type,
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)
    df_stim = _compute_ef_sf(load_stimulation_data(fn_cgi))
    df_stim = df_stim.loc[df_stim["subjid"] != "subj-cat-010-001"].copy()
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    # No fsavg_overlays filter: these are volumetric simulation files.
    mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc.hdf5",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    missing = {"full_file_path", "session_num"} - set(mapped.columns)
    if missing:
        raise ValueError(f"Missing mapped columns: {sorted(missing)}")

    # Preserve the current representative-session definition.
    mapped = mapped.loc[mapped["session_num"] == 2].copy()
    if mapped.empty:
        raise ValueError(f"No eligible session 2 rows for {model_type}, {step}, threshold={threshold}")
    if mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths in mapped sessions.")

    # Recover metadata by exact simulation path, not clinical subject ID.
    lookup = df_query[query_columns].drop_duplicates().rename(columns={"subject_dir": "modeled_subjid"})
    if lookup.isna().any().any():
        raise ValueError("Missing simulation or FreeSurfer metadata in the query lookup.")
    if lookup["fs_subjects_dir"].astype(str).str.strip().eq("").any():
        raise ValueError("Blank FreeSurfer SUBJECTS_DIR in the query lookup.")
    if lookup["full_file_path"].duplicated().any():
        raise ValueError("A query file path has conflicting subject, dataset, or FreeSurfer metadata.")

    metadata_cols = ["modeled_subjid", "dataset_root", "fs_subjects_dir"]
    temporary = {col: f"_query_{col}" for col in metadata_cols}
    reserved = {*temporary.values(), "_query_match"} & set(mapped.columns)
    if reserved:
        raise ValueError(f"Reserved metadata columns already exist: {sorted(reserved)}")

    mapped = mapped.merge(
        lookup.rename(columns=temporary),
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
        indicator="_query_match",
    )

    unmatched = mapped["_query_match"].ne("both")
    if unmatched.any():
        paths = mapped.loc[unmatched, "full_file_path"].unique().tolist()
        raise ValueError(f"Mapped paths do not exactly match df_query.full_file_path: {paths}")

    # Retain matching existing metadata, fill missing metadata, and reject conflicts.
    for col in metadata_cols:
        expected = mapped.pop(temporary[col])
        if col in mapped.columns:
            present = mapped[col].notna()
            if mapped.loc[present, col].ne(expected.loc[present]).any():
                raise ValueError(f"Mapped {col!r} conflicts with metadata from the query inventory.")
        mapped[col] = expected
    mapped = mapped.drop(columns="_query_match")

    # Catch duplicate/missing course identifiers before reading meshes.
    course_keys = ["person_id", "series_num"]
    for label, table in (("Mapped sessions", mapped), ("CGI table", df_cgi)):
        missing = set(course_keys) - set(table.columns)
        if missing:
            raise ValueError(f"{label} is missing course identifiers: {sorted(missing)}")
        if table[course_keys].isna().any().any():
            raise ValueError(f"{label} contains missing course identifiers.")
        if table.duplicated(course_keys).any():
            raise ValueError(f"{label} contains duplicate person_id/series_num combinations.")

    # Repeated courses can reuse one simulation; extract each unique file once.
    inventory = mapped[["full_file_path", *metadata_cols]].drop_duplicates()
    if inventory["full_file_path"].duplicated().any():
        raise ValueError("One simulation file has multiple metadata assignments.")

    roi_frames = []
    for fs_subjects_dir, group in inventory.groupby("fs_subjects_dir", sort=True, observed=True):
        paths = group["full_file_path"].tolist()
        summary = build_volume_roi_summary(
            hdf5_files=paths,
            subjects_dir=fs_subjects_dir,
            subject_ids=group["modeled_subjid"].tolist(),
            simulation_ids=paths,
            atlas_name="aparc",
            field_name="magnE_mean",
            percentile=float(percentile),
            min_elements_per_roi=20,
            freesurfer_lut='/projects/p32903/Alex2/datasets/freesurfer_templates/FreeSurferColorLUT.txt',
            negative_atol=negative_atol,

        )

        if summary.parcel_mean_e is None:
            raise ValueError(f"No ROI means returned for {fs_subjects_dir}")

        means = summary.parcel_mean_e
        percentiles = summary.p95(percentile_weighting)
        if not means.index.equals(percentiles.index) or not means.columns.equals(percentiles.columns):
            raise ValueError("ROI mean and percentile tables do not have matching simulations and ROIs.")
        if means.index.has_duplicates or set(means.index) != set(paths):
            raise ValueError("Extracted simulation IDs do not match the requested unique file paths.")

        means = means.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_mean")
        percentiles = percentiles.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_percentile")
        group_roi = pd.concat([means, percentiles], axis=1)
        group_roi = group_roi.rename_axis("full_file_path").reset_index()

        if not group_roi.columns.is_unique:
            raise ValueError("Duplicate ROI column names after FreeSurfer label normalization.")
        if roi_frames and not group_roi.columns.equals(roi_frames[0].columns):
            raise ValueError(f"ROI columns differ across FreeSurfer datasets: {fs_subjects_dir}")
        roi_frames.append(group_roi)

    df_roi = pd.concat(roi_frames, ignore_index=True)
    if df_roi["full_file_path"].duplicated().any():
        raise ValueError("Duplicate simulation paths in extracted ROI metrics.")

    if combine_hemis:
        lh = {col[3:]: col for col in df_roi.columns if col.startswith("lh.")}
        rh = {col[3:]: col for col in df_roi.columns if col.startswith("rh.")}
        if not lh or set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROIs: {sorted(set(lh) ^ set(rh))}")

        # Average matching summaries; this is not a pooled bilateral percentile.
        bilateral = pd.DataFrame({
            roi: (df_roi[lh[roi]] + df_roi[rh[roi]]) / 2
            for roi in sorted(lh)
        }, index=df_roi.index)
        df_roi = pd.concat([df_roi[["full_file_path"]], bilateral], axis=1)

    roi_columns = df_roi.columns.drop("full_file_path").tolist()
    for label, table in (("mapped sessions", mapped), ("CGI table", df_cgi)):
        overlap = set(roi_columns) & set(table.columns)
        if overlap:
            raise ValueError(f"ROI columns conflict with {label}: {sorted(overlap)}")

    df_stim_roi = mapped.merge(df_roi, on="full_file_path", how="left", sort=False, validate="many_to_one")
    if df_stim_roi[roi_columns].isna().any().any():
        raise ValueError("Missing ROI metrics after merging simulation files.")

    shared_metadata = [col for col in ("age", "sex", "dob") if col in df_stim_roi and col in df_cgi]
    return df_stim_roi.drop(columns=shared_metadata).merge(
        df_cgi,
        on=course_keys,
        how="left",
        sort=False,
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )


In [19]:
"""Run in the existing analysis notebook after defining the preparation/correlation helpers.

Required notebook definitions:
    fn_query, fn_cgi, load_query_results, _prepare_df_efield_volume,
    run_roi_correlations (and its run_efield_correlation dependency).

The preparation helper must use freesurfer_lut=<path>, not lut=<path>.
This script does not change the extraction, cohort selection, or correlation estimator.
"""

from datetime import datetime
from hashlib import sha256
from itertools import product
from numbers import Real
from pathlib import Path
import time

import numpy as np
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm


def _save_correlation_workbook(tables: dict[str, pd.DataFrame], output: Path) -> None:
    """Write tables with filters, frozen headers, and display-only numeric formatting."""
    temporary = output.with_name(f"{output.stem}.tmp.xlsx")
    with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name, index=False, na_rep="")
            sheet = writer.sheets[name]
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            sheet.row_dimensions[1].height = 30

            for cell in sheet[1]:
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="234E70")
                cell.alignment = Alignment(wrap_text=True, vertical="center")

            for j, column in enumerate(table.columns, start=1):
                width = min(36, max(14, len(column) + 2))
                if column in {"roi", "roi_column", "x", "config_label"}:
                    width = 38
                sheet.column_dimensions[get_column_letter(j)].width = width
                number_format = "0.000E+00" if column in {"p", "p_bh_sweep"} else "0.000"
                if column in {"r", "r_abs", "p", "p_bh_sweep", "prepare_seconds"}:
                    for cells in sheet.iter_rows(min_row=2, min_col=j, max_col=j):
                        cells[0].number_format = number_format

            if name == "settings":
                sheet.column_dimensions["A"].width = 30
                sheet.column_dimensions["B"].width = 60
                for cells in sheet.iter_rows(min_row=2):
                    cells[1].alignment = Alignment(wrap_text=True, vertical="top")
                    sheet.row_dimensions[cells[0].row].height = 45

    temporary.replace(output)


# Keep these atlas/ASF labels consistent with the current preparation function.
atlas_name = "aparc"
asf = "asf_5"
percentiles = [95]  # For example: [90, 95, 99].
percentile_weightings = [ "spatial_weighted"] # also "unweighted"
methods = ["pearson", "spearman"] # also "pearson"
combine_hemis_options = [False] # also False

output_dir = Path("results/correlations") / atlas_name
output_dir.mkdir(parents=True, exist_ok=True)
xlsx_path = output_dir / "roi_efield.xlsx"
csv_path = output_dir / "roi_efield.csv"
checkpoint_path = output_dir / "roi_efield_checkpoint.csv"

# Avoid mixing a new run with an older run or partial checkpoint.
for path in (xlsx_path, csv_path, checkpoint_path):
    if path.exists():
        raise FileExistsError(f"Output already exists; use a fresh folder or move the previous run: {path}")

for name, values in (
    ("percentiles", percentiles), ("percentile_weightings", percentile_weightings),
    ("methods", methods), ("combine_hemis_options", combine_hemis_options),
):
    if not values or len(set(values)) != len(values):
        raise ValueError(f"{name} must be nonempty and contain no duplicates.")
if any(isinstance(p, (bool, np.bool_)) or not isinstance(p, Real)
       or not np.isfinite(p) or not 0 <= p <= 100 for p in percentiles):
    raise ValueError("percentiles must contain finite numbers in [0, 100].")
if not set(percentile_weightings) <= {"unweighted", "spatial_weighted"}:
    raise ValueError("Unsupported percentile weighting.")
if not set(methods) <= {"pearson", "spearman"}:
    raise ValueError("Unsupported correlation method.")
if not all(isinstance(value, (bool, np.bool_)) for value in combine_hemis_options):
    raise TypeError("combine_hemis_options must contain booleans.")

# Select all simulation configurations actually present for the fixed filters.
df_query = load_query_results(fn_query)
config_columns = ["model_type_dir", "step", "threshold"]
required = {*config_columns, "asf", "subject_dir", "full_file_path"}
missing = required - set(df_query.columns)
if missing:
    raise ValueError(f"Missing query columns: {sorted(missing)}")

available = df_query.loc[
    df_query["asf"].eq(asf)
    & df_query["subject_dir"].ne("subj-cat-010-001")
    & df_query["full_file_path"].str.endswith("/tdcs_uq_gpc.hdf5", na=False)
].copy()

# only keep these for now, then run on HPC to include all 
keep_steps = ['step_0', 'step_2', 'step_4']
available  = available[available['step'].isin(keep_steps)]

if available.empty or available[config_columns].isna().any().any():
    raise ValueError("No eligible HDF5 configurations, or missing simulation configuration values.")
threshold_values = pd.to_numeric(available["threshold"], errors="raise")
if (available["threshold"].map(lambda value: isinstance(value, (bool, np.bool_))).any()
        or not np.isfinite(threshold_values).all() or threshold_values.mod(1).ne(0).any()):
    raise ValueError("Simulation thresholds must be finite integers.")
available["threshold"] = threshold_values.astype(int)

base_configs = list(
    available[config_columns].drop_duplicates().sort_values(config_columns).itertuples(index=False, name=None)
)
configs = [
    (*base, percentile, combine_hemis, weighting)
    for base, percentile, combine_hemis, weighting in product(
        base_configs, percentiles, combine_hemis_options, percentile_weightings
    )
]

# Means run once per simulation/hemisphere/method, not once per percentile/weighting.
total_batches = (len(base_configs) * len(combine_hemis_options) * len(methods)
                 * (1 + len(percentiles) * len(percentile_weightings)))
results, analysis_records = [], []
started_at = datetime.now().astimezone().isoformat(timespec="seconds")

print(f"Simulation configurations: {len(base_configs)}")
print(f"Preparation calls: {len(configs)}")
print(f"Correlation batches: {total_batches}")

with tqdm(total=total_batches, desc="aparc ROI correlations", unit="batch", mininterval=0, miniters=1) as pbar:
    for model_type, step, threshold, percentile, combine_hemis, weighting in configs:
        progress = {
            "model": model_type, "step": step, "threshold": threshold,
            "percentile": percentile, "weighting": weighting, "hemis": combine_hemis,
        }
        pbar.set_postfix(**progress, status="preparing", refresh=True)
        start = time.perf_counter()

        df = _prepare_df_efield_volume(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
            percentile=percentile,
            combine_hemis=combine_hemis,
            percentile_weighting=weighting,
        )
        prepare_seconds = time.perf_counter() - start

        # This identifies the PREPARED course set; n_courses still reports usable pairs per ROI.
        course_keys = df[["person_id", "series_num"]].sort_values(["person_id", "series_num"])
        prepared_cohort_id = sha256(course_keys.to_csv(index=False).encode()).hexdigest()[:16]
        dataset_roots = ";".join(sorted(df["dataset_root"].unique()))
        first_summary = percentile == percentiles[0] and weighting == percentile_weightings[0]
        metrics = ["mean", "percentile"] if first_summary else ["percentile"]

        for metric, method in product(metrics, methods):
            pbar.set_postfix(**progress, metric=metric, method=method, status="correlating", refresh=True)
            corr = run_roi_correlations(df, combine_hemis=combine_hemis, metric=metric, method=method).copy()
            if corr.empty:
                raise ValueError(f"No ROI correlation results for {progress}, metric={metric}, method={method}")

            metric_label = f"p{float(percentile):g}_{weighting}" if metric == "percentile" else "mean"
            hemis_label = "bilateral" if combine_hemis else "separate"
            config_label = f"{model_type}|{step}|thr{threshold}|{metric_label}|{hemis_label}|{method}"
            metadata = {
                "analysis_id": len(analysis_records) + 1,
                "config_label": config_label,
                "atlas": atlas_name,
                "atlas_file": "aparc+aseg.mgz",
                "asf": asf,
                "model_type": model_type,
                "step": step,
                "threshold": int(threshold),
                "metric": metric,
                "percentile": float(percentile) if metric == "percentile" else np.nan,
                "percentile_weighting": weighting if metric == "percentile" else pd.NA,
                "mean_weighting": "unweighted_elements" if metric == "mean" else pd.NA,
                "combine_hemis": bool(combine_hemis),
                "hemisphere_aggregation": "mean_of_hemisphere_summaries" if combine_hemis else "none",
                "method": method,
                "field_name": "magnE_mean",
                "representative_session": 2,
                "dataset_roots": dataset_roots,
                "prepared_cohort_id": prepared_cohort_id,
            }
            corr = corr.assign(**metadata, r_abs=corr["r"].abs())

            # Persist completed batches before the expensive next preparation call.
            corr.to_csv(checkpoint_path, mode="a", header=not checkpoint_path.exists(), index=False)
            results.append(corr)
            analysis_records.append({
                **metadata,
                "n_roi_results": len(corr),
                "n_ok": int(corr["status"].eq("ok").sum()),
                "n_courses_min": int(corr["n_courses"].min()),
                "n_courses_max": int(corr["n_courses"].max()),
                "n_patients_min": int(corr["n_patients"].min()),
                "n_patients_max": int(corr["n_patients"].max()),
                "prepare_seconds": prepare_seconds,
            })
            pbar.update(1)
            pbar.refresh()

        pbar.set_postfix(**progress, status="done", last_sec=round(time.perf_counter() - start, 1), refresh=True)

if len(analysis_records) != total_batches:
    raise RuntimeError("Completed correlation batch count does not match the planned count.")

df_results = pd.concat(results, ignore_index=True)
df_analyses = pd.DataFrame(analysis_records)
test_keys = [
    "atlas", "asf", "model_type", "step", "threshold", "metric", "percentile",
    "percentile_weighting", "combine_hemis", "method", "hemisphere", "roi", "y",
]
if df_results.duplicated(test_keys).any():
    raise ValueError("Duplicate ROI tests in the final results; do not silently deduplicate them.")

# Exploratory BH adjustment across ALL valid tests in this completed sweep.
# This does not correct invalid/unclustered input p-values.
df_results["p_bh_sweep"] = np.nan
valid = df_results["status"].eq("ok")
p_values = pd.to_numeric(df_results.loc[valid, "p"], errors="raise")
if not np.isfinite(p_values).all() or not p_values.between(0, 1).all():
    raise ValueError("An 'ok' correlation has an invalid p-value.")
if valid.any():
    df_results.loc[valid, "p_bh_sweep"] = multipletests(p_values, method="fdr_bh")[1]

first_columns = [
    "analysis_id", "config_label", "atlas", "model_type", "step", "threshold", "metric",
    "percentile", "percentile_weighting", "mean_weighting", "combine_hemis", "hemisphere", "roi",
    "method", "r", "r_abs", "p", "p_bh_sweep", "n_courses", "n_patients", "n_missing", "status",
]
df_results = df_results[first_columns + [c for c in df_results.columns if c not in first_columns]]

settings = {
    "started_at": started_at,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "query_file": str(Path(fn_query).resolve()),
    "clinical_file": str(Path(fn_cgi).resolve()),
    "atlas": atlas_name,
    "atlas_file": "aparc+aseg.mgz",
    "simulation_selection": "All existing HDF5 model/step/threshold combinations for asf_5, excluding subj-cat-010-001",
    "percentiles": str(percentiles),
    "percentile_weightings": str(percentile_weightings),
    "methods": str(methods),
    "combine_hemis_options": str(combine_hemis_options),
    "representative_session": "2; unchanged from the preparation function",
    "min_elements_per_roi": "20; unchanged from the preparation function",
    "outcome": "cgi_change as supplied by the clinical loader; not recomputed",
    "mean_definition": "Unweighted arithmetic mean across tetrahedral elements; analyzed only once per configuration",
    "percentile_definitions": "unweighted: equal elements; spatial_weighted: tetrahedron-volume-weighted empirical CDF",
    "bilateral_definition": "Arithmetic mean of left/right summary values, not a pooled bilateral percentile",
    "mean_parameter_blanks": "percentile and percentile_weighting are blank because neither defines the ROI mean",
    "prepared_cohort_id": "Hash of sorted prepared person_id/series_num pairs; not a hash of all ROI-specific usable pairs",
    "multiplicity": "p_bh_sweep: BH across all finite status=ok tests in this completed atlas sweep",
    "p_value_limitation": "Existing two-sided Pearson/Spearman tests are unchanged; no patient-cluster correction",
    "checkpoint": "Completed batches only; no full-sweep BH adjustment and no automatic resume",
}
df_settings = pd.DataFrame(settings.items(), columns=["setting", "value"])

df_results.to_csv(csv_path, index=False)
_save_correlation_workbook({"results": df_results, "analyses": df_analyses, "settings": df_settings}, xlsx_path)

print(f"Saved {len(df_results)} ROI results from {len(df_analyses)} analysis batches.")
print(f"Excel: {xlsx_path}")
print(f"CSV: {csv_path}")
print(f"Checkpoint: {checkpoint_path}")


Simulation configurations: 18
Preparation calls: 18
Correlation batches: 72


aparc ROI correlations:  50%|█████     | 36/72 [46:58<25:30, 42.51s/batch, hemis=0, model=skin_single, percentile=95, status=preparing, step=step_0, threshold=80, weighting=spatial_weighted]                                        /gpfs/projects/p32903/Alex2/ect-workflow/05_analysis/simnibs_parcel_analysis/atlas_volume.py:268: RuntimeWarning: Field 'magnE_mean': setting 2 negligible negatives to zero; minimum=-4.6825838e-10; tolerance=1e-08
  field, _ = scalar_mesh_field(
aparc ROI correlations:  56%|█████▌    | 40/72 [50:29<17:04, 32.00s/batch, hemis=0, model=skin_single, percentile=95, status=preparing, step=step_0, threshold=90, weighting=spatial_weighted]                                      /gpfs/projects/p32903/Alex2/ect-workflow/05_analysis/simnibs_parcel_analysis/atlas_volume.py:268: RuntimeWarning: Field 'magnE_mean': setting 2 negligible negatives to zero; minimum=-4.6825838e-10; tolerance=1e-08
  field, _ = scalar_mesh_field(
aparc ROI correlations:  61%|██████    | 44/72 [54

ValueError: /gpfs/projects/b32903/Alex/results/asf_5/STU00225089/skin_single/subj-cat-058-001/BL/80/20261007_100833/step_4/tdcs_uq_gpc.hdf5: Field 'magnE_mean' contains 56 values below -1e-08; minimum=-3.0913537e-07

ValueError: Checkpoint labels are not a contiguous prefix of this sweep plan; settings may have changed.